# AutoGluon 101 — from a table to a reusable predictor

**Purpose.** Learn what AutoGluon Tabular automates, build a small classification example, compare it with a logistic-regression baseline, and explain how the artifact can enter a reliable pipeline. This is an **individual, ungraded** learning activity. It supplements the Lab 07 student brief in the separate course checkout; it does not replace its HighRev pipeline, registry or deployment exercises.

**Starting point.** Follow [the setup guide](AUTOGLUON_101_SETUP.md), copy this notebook to your own disposable working directory and select its Python 3.11 kernel. The example uses the locally bundled scikit-learn breast-cancer dataset and needs no data download. It is an educational classification exercise, **not a medical tool**. In this dataset **0 = malignant and 1 = benign**: our positive-class probability and ROC AUC concern the benign label, not the probability of cancer.

**Working agreement.** Run cells in order; write observations and answers yourself. Do not train in the course checkout, reference project, team project or a retained rehearsal. Each run creates a new output directory; never reuse or overwrite a previous predictor. Do not treat the example's metrics as deployment approval.

## The mental model

```text
labeled table → split data → preprocess → choose on validation → compare candidates
      → evaluate the chosen candidate on untouched test → save and reload artifact
```

AutoML can automate feature preprocessing, fitting candidate model families, some selection and ensembling. The engineer still defines the prediction time and target, collects permissible data, chooses splits and metrics, prevents leakage, defines resource limits and decides whether a candidate is safe to release.

**Outcomes:** explain `TabularPredictor`, `fit`, `predict`, `predict_proba`, the leaderboard and saved artifacts; compare with a baseline using the same held-out selection set; describe ensemble/preset tradeoffs; and sketch how this can fit into Kedro or an orchestrator.

## 0. Check the kernel

The Lab 07 environment uses **AutoGluon Tabular 1.3.1 and Python 3.11**. This notebook is written for that version; the 1.6.1 documentation supplied in the roadmap can describe different presets and dependencies. If the check fails, select the kernel from the setup guide rather than installing packages inside the notebook. The extra notebook tools are separate from the Lab 07 project lock.

In [11]:
import importlib.metadata as metadata
import json
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ag_version = metadata.version("autogluon.tabular")
print("Python:", sys.version.split()[0])
print("AutoGluon Tabular:", ag_version)
print("Kernel interpreter:", sys.executable)
if sys.version_info[:2] != (3, 11) or ag_version != "1.3.1":
    raise RuntimeError("Select the Lab 07 notebook kernel: Python 3.11 and AutoGluon Tabular 1.3.1.")

Python: 3.12.13
AutoGluon Tabular: 1.3.1
Kernel interpreter: d:\4.Teaching\ASI\lecture-07\presentation-autogluon\.venv\Scripts\python.exe


RuntimeError: Select the Lab 07 notebook kernel: Python 3.11 and AutoGluon Tabular 1.3.1.

## 1. Load data and set a prediction contract

Each row is a sample with 30 numerical features. The target is named `target`; class `1` is **benign**. We use ROC AUC to compare probability rankings, not to choose a clinical decision threshold. The data is a convenient built-in teaching dataset; a real application must establish data provenance, representativeness, prediction-time feature availability and a suitable metric.

In [12]:
dataset = load_breast_cancer(as_frame=True)
data = dataset.frame.copy()
TARGET = "target"
FEATURES = list(dataset.feature_names)
assert dataset.target_names.tolist() == ["malignant", "benign"]
print("Shape:", data.shape)
print("Number of features:", len(FEATURES))
print("Class labels:", dict(enumerate(dataset.target_names)))
print("Target counts:", data[TARGET].value_counts().sort_index().to_dict())

Shape: (569, 31)
Number of features: 30
Class labels: {0: np.str_('malignant'), 1: np.str_('benign')}
Target counts: {0: 212, 1: 357}


## 2. Split before fitting anything

We reserve the final test partition first, then split the remainder into **fit** and **validation**. Stratification preserves class proportions approximately; it does not handle repeated customers or temporal leakage. Lab 06 uses customer-grouped splits; a real time-dependent problem may instead need a time-based split.

The validation set is given to AutoGluon as `tuning_data`: it participates in internal model selection and ensemble fitting, and is also used below to choose between the AutoGluon candidate and the baseline. Therefore validation scores are **selection evidence**, not an unbiased final evaluation. Depending on fit options, AutoGluon may use tuning data in the final model. The test partition must never be passed to `fit`, `leaderboard` for selection, or preprocessing fit.

In [13]:
SEED = 42
fit_and_validation, test = train_test_split(
    data, test_size=0.20, random_state=SEED, stratify=data[TARGET]
)
fit, validation = train_test_split(
    fit_and_validation, test_size=0.25, random_state=SEED,
    stratify=fit_and_validation[TARGET],
)
assert not (set(fit.index) & set(validation.index))
assert not (set(fit.index) & set(test.index))
assert not (set(validation.index) & set(test.index))
print({"fit": len(fit), "validation": len(validation), "test": len(test)})

{'fit': 341, 'validation': 114, 'test': 114}


## 3. Create a fresh run directory

The run root is relative to the directory in which you launched the notebook. That is why the setup guide creates a separate working directory. `mkdtemp` creates a new, uniquely named directory; the model path must not already exist. Keep the path for inspection and recovery. It is a local run, not a registry or a production promotion.

A fixed split seed helps repeat the partition, but it does not guarantee identical training results across versions, hardware, time budgets or scheduling. Record the input identity, package versions, settings and decisions along with the artifact. The supplemental Jupyter environment is not fully locked by the Lab 07 lock.

In [14]:
run_root = Path.cwd() / "autogluon-101-runs"
run_root.mkdir(exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="run-", dir=run_root))
model_path = run_dir / "autogluon-model"
assert not model_path.exists()
print("Run directory:", run_dir.resolve())

Run directory: D:\4.Teaching\ASI\highrev-mlops-course\sessions\07_autogluon\autogluon-101-runs\run-o616enj4


## 4. Fit a simple baseline

A baseline gives the AutoML result a meaningful comparison. The scikit-learn pipeline fits the scaler **only on fit data**, then uses the same fitted scaler for validation and inference. Positive-class probabilities are selected by the actual class label rather than assuming an arbitrary column position.

In [ ]:
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED))
baseline.fit(fit[FEATURES], fit[TARGET])
positive_index = list(baseline.classes_).index(1)
baseline_val_proba = baseline.predict_proba(validation[FEATURES])[:, positive_index]
baseline_val_auc = roc_auc_score(validation[TARGET], baseline_val_proba)
print(f"Baseline validation ROC AUC: {baseline_val_auc:.4f}")

Baseline validation ROC AUC: 0.9980


## 5. Fit the first AutoGluon candidate

`label` names the target, `eval_metric` tells AutoGluon what to optimize, and `path` names the new artifact. `fit` receives only fit and validation data. This deliberately small CPU example asks for random forest (`RF`) and extra trees (`XT`) and allows a weighted ensemble. It is a bounded portfolio, **not exhaustive hyperparameter optimization**. The budget is approximate, not a hard total deadline; a model or ensemble may be skipped or fail under resource limits. No GPU or extra LightGBM/CatBoost installation is required for these requested families.

`medium_quality` is the explicit 1.3.1 preset. The version-specific aliases `medium` and `best` also exist, but `best_quality` may use substantially more time and memory. Explicit options below limit this first example; changing only the preset while keeping them may not demonstrate the preset's full default behavior. Do not run a stronger preset during the final test audit.

In [16]:
fit_started = time.perf_counter()

predictor = TabularPredictor(
    label=TARGET,
    problem_type="binary",
    eval_metric="roc_auc",
    positive_class=1,
    path=str(model_path),
).fit(
    train_data=fit,
    tuning_data=validation,
    presets="medium_quality",
    hyperparameters={"RF": {"n_estimators": 50}, "XT": {"n_estimators": 50}},
    num_bag_folds=0,
    num_stack_levels=0,
    num_cpus=1,
    time_limit=90,
)
print("Trained models:", predictor.model_names())
print("Class labels:", predictor.class_labels, "positive:", predictor.positive_class)
training_seconds = time.perf_counter() - fit_started
artifact_bytes = sum(p.stat().st_size for p in model_path.rglob("*") if p.is_file())
print(f"AutoGluon elapsed fit: {training_seconds:.2f} seconds; artifact: {artifact_bytes} bytes")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          28
Memory Avail:       3.66 GB / 31.84 GB (11.5%)
Disk Space Avail:   1594.61 GB / 1907.71 GB (83.6%)
Presets specified: ['medium_quality']
Failed to save metadata file due to exception No module named 'pkg_resources', skipping...
Beginning AutoGluon training ... Time limit = 90s
AutoGluon will save models to "d:\4.Teaching\ASI\highrev-mlops-course\sessions\07_autogluon\autogluon-101-runs\run-o616enj4\autogluon-model"
Train Data Rows:    341
Train Data Columns: 30
Tuning Data Rows:    114
Tuning Data Columns: 30
Label Column:       target
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Availa

Trained models: ['RandomForest', 'ExtraTrees', 'WeightedEnsemble_L2']
Class labels: [np.int64(0), 1] positive: 1
AutoGluon elapsed fit: 0.47 seconds; artifact: 533528 bytes


## 6. Predictions and the leaderboard

`predict()` returns class labels; `predict_proba()` returns probabilities. With a binary DataFrame output, columns use the original class labels. Here `1` is benign. `predictor.leaderboard(validation)` displays scores for the supplied **validation** frame: despite the column name `score_test`, it is not our untouched test set. `score_val` is AutoGluon's internal validation score. Both can reflect the same supplied tuning data; compare definitions, not only column names.

A weighted ensemble may or may not appear and can select only one candidate. Examine the actual rows and latency columns. Repeatedly trying configurations on the same validation set can overfit your decisions. Never use the final test leaderboard to pick a winner.

In [17]:
sample = validation[FEATURES].iloc[:5]
labels = predictor.predict(sample)
probabilities = predictor.predict_proba(sample)
print("Predicted labels:", labels.to_list())
print("Probability columns:", probabilities.columns.to_list())
print(probabilities)
leaderboard = predictor.leaderboard(validation, silent=True)
display(leaderboard)

Predicted labels: [1, 0, 0, 1, 1]
Probability columns: [0, 1]
          0       1
544  0.0525  0.9475
56   1.0000  0.0000
27   0.9650  0.0350
139  0.0175  0.9825
109  0.0350  0.9650


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.992630,0.992630,roc_auc,0.034488,0.035527,0.223698,0.001000,0.000000,0.007005,2,True,3
1,ExtraTrees,0.992466,0.992466,roc_auc,0.016951,0.017512,0.063554,0.016951,0.017512,0.063554,1,True,2
2,RandomForest,0.983623,0.983623,roc_auc,0.016537,0.018014,0.153140,0.016537,0.018014,0.153140,1,True,1


## 7. Compare and freeze a choice

Both candidates see the same validation cases and metric. This does **not** make the procedures identical: AutoGluon can choose among multiple fitted candidates using that same data, whereas the baseline is one fixed model. Thus the comparison is a model-selection exercise, not a final unbiased claim. The simple rule below chooses the higher validation AUC; a real gate should also include justified absolute quality, latency, artifact size and other requirements.

In [18]:
ag_val_proba = predictor.predict_proba(validation[FEATURES])[1].to_numpy()
ag_val_auc = roc_auc_score(validation[TARGET], ag_val_proba)
comparison = pd.DataFrame([
    {"candidate": "logistic_regression", "validation_roc_auc": baseline_val_auc},
    {"candidate": "autogluon", "validation_roc_auc": ag_val_auc},
]).sort_values("validation_roc_auc", ascending=False).reset_index(drop=True)
display(comparison)
winner = str(comparison.loc[0, "candidate"])
print("Choice frozen before test:", winner)
manifest = {
    "dataset": "sklearn.load_breast_cancer (built-in educational dataset)",
    "target": TARGET, "positive_label": 1, "split_seed": SEED,
    "python": sys.version.split()[0], "autogluon": ag_version,
    "scikit_learn": metadata.version("scikit-learn"),
    "pandas": metadata.version("pandas"),
    "autogluon_models_requested": ["RF", "XT"],
    "time_limit_seconds": 90, "num_cpus": 1,
    "fit_rows": len(fit), "validation_rows": len(validation), "test_rows": len(test),
    "observed_fit_seconds": training_seconds, "autogluon_artifact_bytes": artifact_bytes,
    "baseline_validation_auc": float(baseline_val_auc),
    "autogluon_validation_auc": float(ag_val_auc),
    "chosen_candidate": winner,
}
(run_dir / "run-manifest.json").write_text(
    json.dumps(manifest, indent=2, allow_nan=False) + "\n", encoding="utf-8"
)
print("Saved run manifest:", run_dir / "run-manifest.json")

,candidate,validation_roc_auc
0,logistic_regression,0.998035
1,autogluon,0.992630


Choice frozen before test: logistic_regression
Saved run manifest: d:\4.Teaching\ASI\highrev-mlops-course\sessions\07_autogluon\autogluon-101-runs\run-o616enj4\run-manifest.json


## 8. Evaluate the chosen candidate once on the test set

The `test-consumed.json` marker below prevents accidentally rerunning this cell **within this run directory**. It is written before test evaluation, even if evaluation then fails; preserve the failure and investigate. This is a teaching guard, not a security boundary: creating another run with the same split does not make the same test set unseen again. Do not use these results to go back and reselect a model.

Accuracy and F1 below use an illustrative probability threshold of 0.5 and positive class **benign**. AUC measures ranking across thresholds. A production threshold and positive class must follow the problem's actual costs and use case.

In [19]:
marker = run_dir / "test-consumed.json"
with marker.open("x", encoding="utf-8") as stream:
    json.dump({"winner": winner, "status": "test access started"}, stream, indent=2)
if winner == "autogluon":
    test_proba = predictor.predict_proba(test[FEATURES])[1].to_numpy()
else:
    test_proba = baseline.predict_proba(test[FEATURES])[:, positive_index]
test_labels = (test_proba >= 0.5).astype(int)
results = {
    "winner": winner,
    "positive_label": "benign (1)",
    "test_roc_auc": float(roc_auc_score(test[TARGET], test_proba)),
    "test_accuracy_at_0_5": float(accuracy_score(test[TARGET], test_labels)),
    "test_f1_benign_at_0_5": float(f1_score(test[TARGET], test_labels, pos_label=1)),
}
print(json.dumps(results, indent=2))
(run_dir / "test-results.json").write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")

{
  "winner": "logistic_regression",
  "positive_label": "benign (1)",
  "test_roc_auc": 0.9950396825396824,
  "test_accuracy_at_0_5": 0.9824561403508771,
  "test_f1_benign_at_0_5": 0.9861111111111112
}


203

## 9. Save, reload and reason about artifacts

AutoGluon saves its fitted predictor under `model_path` during training. Reload that **candidate** and compare its probabilities on a few validation rows. This demonstrates persistence; it does not make that candidate the selected winner or an approved MLflow champion. Loading uses pickle: load only artifacts you trust and use compatible Python and AutoGluon versions.

In [20]:
reloaded = TabularPredictor.load(str(model_path))
before = predictor.predict_proba(sample)[1].to_numpy()
after = reloaded.predict_proba(sample)[1].to_numpy()
np.testing.assert_allclose(before, after, rtol=1e-6, atol=1e-8)
print("Reloaded candidate matches on", len(sample), "validation rows.")
print("Artifact:", model_path.resolve())

This could mean that the predictor was fit in a version `<=0.5.2`.


ModuleNotFoundError: No module named 'pkg_resources'

For the optional deployment optimization `predictor.save_space(remove_data=True)`, wait until you no longer need analyses that depend on cached training/validation data. It modifies the saved predictor and can disable some advanced operations. It removes cached tables, but a fitted model can still retain information about its training data. The notebook does **not** call it automatically or claim an artifact is safe to publish.

To retain the baseline for a later application, use an approved, versioned artifact store and a trusted serialization format compatible with the serving environment; this notebook keeps the baseline in memory. The Lab 07 serving exercise uses a separate artifact, API, container and explicit version identity.

## 10. How AutoGluon works and when to use it

- **Preprocessing:** infers and transforms feature types; inspect what happened and ensure inference receives the same contract.
- **Model families:** can fit different algorithms and compare them using a selected metric and evaluation protocol. A single family, as configured in the deployment demo, is a deliberately narrow example.
- **Bagging:** trains multiple models on resampled/folded data; out-of-fold predictions can help build later layers. It costs time, memory and storage.
- **Stacking:** trains a subsequent model on earlier models' predictions, taking care not to feed in-sample predictions as if they were out-of-fold.
- **Weighted ensembles:** combine candidate predictions with learned weights; they need not improve every problem and can increase serving cost.
- **Presets and budget:** change the search strategy and resource/quality tradeoff. A budget is not a strict wall-clock deadline. More search does not guarantee a better deployed model.
- **HPO:** exploring hyperparameter search spaces is distinct from fitting several model families. In the documented 1.3.1 default workflow, extensive HPO is not automatically exhaustive or the default. Start with a suitable preset and compare under your constraints.

Use AutoGluon as a candidate generator when tabular data, compute budget and operational requirements fit. Consider a simpler model or another approach when interpretability, strict latency, tiny memory, specialized data or strong constraints dominate. Always establish a simple baseline and define evaluation independently of the leaderboard.

## 11. Engineering bridge: Kedro first, Airflow later

In the course, keep the existing Kedro data-processing nodes and checks. A minimal integration can pass a **prepared training dataset and configuration** into a training node, then register a durable output artifact and metrics; a separate prediction node loads a trusted pinned artifact and validates the input contract. Avoid putting hidden paths or mutable registry state inside a pure transformation node.

```text
versioned input → existing Kedro preparation → AutoGluon training node
                → model artifact + metrics → quality decision → serving image
```

Airflow is an optional scheduling/orchestration extension: a DAG can coordinate independently executable preparation, training, evaluation and deployment tasks, with durable artifacts passed by reference. It does not replace testable model code, storage, data validation or a model-quality gate. This is an architecture sketch, not an implemented Airflow DAG or claim that this notebook updates the HighRev project.

## 12. Your individual practice and questions

1. Why did we reserve a test set before training? What changes if validation is reused for many searches?
2. What do `label="target"`, `predict()` and `predict_proba()` mean? Which class is positive here?
3. Inspect the leaderboard: which models actually trained? Is there a weighted ensemble? What happens if one family is skipped?
4. Compare candidate scores, latency and artifact size. Does the best AUC necessarily make the best deployed model?
5. Explain a preprocessing or split that would cause leakage. Why does Lab 06 group customers instead of using this toy row split?
6. What would `best_quality` change in the **1.3.1** docs? Which explicit settings in this notebook would constrain that experiment?
7. Sketch a Kedro node's inputs and outputs and the tests you would write. Which work must remain outside the node?
8. Explain model persistence, trusted artifacts, version compatibility and one missing-model failure to another engineer.

**Optional extension:** start a new run directory and compare a bounded different configuration using validation only. Preserve both artifacts; do not repeatedly consult the same final test set as you iterate. The marker protects one directory, not the experiment's overall evaluation protocol.

## Optional four-week progression

This optional self-study roadmap assumes roughly 5 to 7 hours per week, as a planning estimate rather than a Lab 07 requirement.

| Week | Focus | Evidence of learning |
|---|---|---|
| 1 | Tabular basics: data, predictor, scores and baseline | Run and explain this notebook independently. |
| 2 | Model families, preprocessing, presets, bagging, stacking and leakage | Explain the leaderboard and compare a bounded validation experiment. |
| 3 | Reusable functions, artifact management and orchestration | Propose a tested Kedro integration; study Airflow as an optional scheduler. |
| 4 | Teach and rehearse | Prepare a practical demonstration, anticipate failure cases and explain design decisions. |

**Read next:** [AutoGluon 1.3.1 quick start](https://auto.gluon.ai/1.3.1/tutorials/tabular/tabular-quick-start.html), [how it works](https://auto.gluon.ai/1.3.1/tutorials/tabular/how-it-works.html), [fit API](https://auto.gluon.ai/1.3.1/api/autogluon.tabular.TabularPredictor.fit.html), [deployment optimization](https://auto.gluon.ai/1.3.1/tutorials/tabular/advanced/tabular-deployment.html), and [scikit-learn dataset documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html). AutoGluon documentation is versioned; use the matching version when checking behavior.